# M2 Notebook Release Header
**Release Header (Auto)**
- Purpose: analysis workflow notebook.
- Inputs: local source tables and matrices.
- Outputs: figures and summary metrics.
- Dependencies: existing scientific Python packages only.
- Execution Order: run top-to-bottom.


# Publish Notebook Header

- Purpose: Reproducible analysis notebook for M2 release package.
- Inputs: Local project data files (configured via relative paths or project root variable).
- Outputs: Analysis tables and figures with unchanged logic/style.
- Execution Order: Run cells top-to-bottom.
- Privacy: Local identity/path tokens are anonymized for release.


-lolipop_chart_part1 -main


## Section: Core Analysis
This section preserves original computation and visualization behavior.


In [6]:
########################
# This is the loopable version of lolipopchart part1,
#please do not split the main block

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp
from scipy.optimize import curve_fit
import os
import re

%matplotlib inline

In [7]:
def frequenceTransform(df_signal_dic,df_signal_id):
    ## test1: 20hz to 10hz
    ## test2: 30hz to 10hz

    resized_pd = {}

    for i in df_signal_id:
        if len(df_signal_dic[i]["C"])%3==1:
            temp_dic = df_signal_dic[i]["C"][:-1]

            temp = temp_dic.values
            temp = np.reshape(temp,(int(len(temp)/3),3))

            new_table = np.zeros(len(temp))
            for j in range(len(temp)):
                new_table[j] = temp[j].mean()
            new_table = pd.DataFrame(new_table)

            resized_pd[i] = new_table
        elif len(df_signal_dic[i]["C"])%3==2:
            temp_dic = df_signal_dic[i]["C"][:-2]

            temp = temp_dic.values
            temp = np.reshape(temp,(int(len(temp)/3),3))

            new_table = np.zeros(len(temp))
            for j in range(len(temp)):
                new_table[j] = temp[j].mean()
            new_table = pd.DataFrame(new_table)

            resized_pd[i] = new_table
        else:
            temp_dic = df_signal_dic[i]["C"]

            temp = temp_dic.values
            temp = np.reshape(temp,(int(len(temp)/3),3))

            new_table = np.zeros(len(temp))
            for j in range(len(temp)):
                new_table[j] = temp[j].mean()
            new_table = pd.DataFrame(new_table)

            resized_pd[i] = new_table
# test downsampleing
    plt.figure(figsize=(15, 6))


    plt.subplot(2, 1, 1)
    before_data = df_signal_dic[1]
    plt.plot(np.arange(len(before_data['index'])), before_data['C'], 'b-', linewidth=0.8, label='before')
    plt.title('before (unit_id=0)')
    plt.xlabel('frame')
    plt.ylabel('signal')

    plt.grid(alpha=0.3)
    plt.legend()

    
    plt.subplot(2, 1, 2)
    after_data = resized_pd[1]
    plt.plot(np.arange(len(after_data)), after_data, 'r-', linewidth=0.8, label='after')
    plt.title('after (unit_id=0)')
    plt.xlabel('frame')
    plt.ylabel('signal')
    plt.grid(alpha=0.3)
    plt.legend()


    plt.tight_layout()
    plt.show()
    plt.close()
    return resized_pd


In [8]:
def Binary_equation(x,a,b,c):
    return a*(x-b)**2+c

In [9]:
# path_signal = r'D:/XJN/behavior_signaling/CamkII TST/10/save_result'
# path_behaviour = r'D:/XJN/behavior_signaling/CamkII TST/10'

# save_path = r'D:/RQ/M2 Project/summary/result/CamkII TST/10/m2_v26_lollipop_chart_part1'
# fig_path = r'D:/RQ/M2 Project/summary/result/CamkII TST/10/m2_v26_lollipop_chart_part1'

-main


In [ ]:
#VIP:1,2,3,4;CamkII_TST,,1~10; SST:1~3 PV:1~7;Hsyn:1~5
# Warpper
#Y:LH0167,CSDS0179,CSDS0126,CSDS0087
cell_type = 'pre'
IDs = ['LH5779','CSDS0103','CSDS0370']
model = 'test2'
test = 'TST' # should not be used in test1
projectpath = '/home/user_theft/XJN_M2project'
summary_path = '/home/user_theft/XJN_M2project/summary'
signal_dir = projectpath + '/signal_data/'
behavioral_dir = projectpath + '/behavioral_data/'
##############
#Main function
##############
for ID in IDs:
    print('Analysis '+cell_type + '_' + ID +':')
    signal_path = os.path.join(signal_dir,model,cell_type,ID,test)
    behavioral_path = os.path.join(behavioral_dir,model,cell_type,ID)
    if os.path.exists(signal_path)==False:
        print('Signal File not found, skipping ' + cell_type + '_' + ID)
        continue
    elif os.path.exists(behavioral_path)==False:
        print('Behavioral File not found, skipping ' + cell_type + '_' + ID)
        continue

    df_signal = pd.read_csv(os.path.join(signal_path, 'C.csv'))
    df_raw = pd.read_csv(os.path.join(signal_path, 'YrA.csv'))

    df_im = pd.read_csv(os.path.join(behavioral_path, 'im.csv'))
    df_behaviour = pd.read_csv(os.path.join(behavioral_path, 'Area_Change.csv'))

    save_path = os.path.join(summary_path,model,cell_type,ID,test,'signal_save')
    if not os.path.exists(save_path):os.makedirs(save_path)

    ##################################### 
    # preprocessing(read&cut)
    ####################################
    column_list = ['unit_id','frame','C']
    df_signal = df_signal[column_list]
    column_list_YrA = ['unit_id','frame','YrA']
    df_raw = df_raw[column_list_YrA]

    df_raw = df_raw.rename(columns={'YrA':'C'})
    df_signal_id = list(df_signal['unit_id'].values)
    df_signal_id = list(set(df_signal_id))
    print('Signal ID count:')

    print(len(df_signal_id))


    df_signal_dic = {}
    for i in df_signal_id:
        df_signal_dic[i] = df_signal[df_signal['unit_id']==i].reset_index()
        
    df_raw_dic = {}
    for i in df_signal_id:
        df_raw_dic[i] = df_raw[df_raw['unit_id']==i].reset_index()

    #limit length
    resized_df_signal_dic = frequenceTransform(df_signal_dic,df_signal_id)
    resized_df_raw_dic = frequenceTransform(df_raw_dic,df_signal_id)
    df_im = df_im.T.reset_index().drop(columns=['index'])
    df_behaviour = df_behaviour.T.reset_index().drop(columns=['index'])
    print('Data raw size:')
    print(len(df_im))
    print(len(df_behaviour))
    print(len(resized_df_signal_dic[1]))
    print(len(resized_df_raw_dic[1]))

    min_frame = min(len(resized_df_signal_dic[1]),len(resized_df_raw_dic[1]),len(df_im),len(df_behaviour))

    df_im_update = df_im[:min_frame]
    df_behaviour_update = df_behaviour[:min_frame]

    resized_df_signal_dic_update = {}
    for i in df_signal_id:
        temp_table = resized_df_signal_dic[i][:min_frame]
        resized_df_signal_dic_update[i] = temp_table

    resized_df_raw_dic_update = {}
    for i in df_signal_id:
        temp_table = resized_df_raw_dic[i][:min_frame]
        resized_df_raw_dic_update[i] = temp_table
    print('Data updated size:')
    print(len(df_im_update))
    print(len(df_behaviour_update))
    print(len(resized_df_signal_dic_update[1]))
    print(len(resized_df_raw_dic_update[1]))

    ax = plt.figure(figsize=(12,3))
    plt.plot(df_im_update)
    plt.title(cell_type + ': ' + ID + ' im')
    plt.show()
    plt.close()

    ##################################### 
    # select efficient transition
    ####################################
    S_to_I = []
    for i in range(len(df_im_update)-1):
        if (df_im_update[0][i+1]-df_im_update[0][i]) == 1:
            S_to_I.append(i)
        else:
            continue
    print('S_to_I:')
    print(S_to_I)

    # efficient transition select
    S_to_I_update = []
    df_im_update_temp = df_im_update.copy()
    for i in S_to_I:
        if i < 20 or i > (len(df_im_update_temp)-20):
            continue
        else:
            temp_backward =  df_im_update_temp[(i-20):i]  
            temp_b_values = temp_backward.sum().values

            temp_forward =  df_im_update_temp[i:i+20]   
            temp_f_values = temp_forward.sum().values

            if temp_b_values[0]==0:
                if temp_f_values[0]==19:
                    S_to_I_update.append(i)
                else:
                    for num in range(i+1,i+20):
                        if df_im_update_temp.loc[num,0]==1:
                            df_im_update_temp.loc[num,0]=0
                        else:
                            break
                    
            else:
                continue
    print(S_to_I_update)

    I_to_S = []
    for i in range(len(df_im_update)-1):
        if (df_im_update.loc[i+1,0]-df_im_update.loc[i,0]) == -1:
            I_to_S.append(i)
        else:
            continue
    print('I_to_S:')
    print(I_to_S)

    # efficient transition select
    I_to_S_update = []
    df_im_update_temp = df_im_update.copy()
    for i in I_to_S:
        if i < 20 or i > (len(df_im_update_temp)-20):
            continue
        else:
            temp_backward =  df_im_update_temp[(i-20):i]  
            temp_b_values = temp_backward.sum().values

            temp_forward =  df_im_update_temp[i:i+20]   
            temp_f_values = temp_forward.sum().values

            if temp_b_values[0]==20: 
                if temp_f_values[0]==1:
                    I_to_S_update.append(i)
                else:
                    for num in range(i+1,i+20):           
                        if df_im_update_temp.loc[num,0]==0:
                            df_im_update_temp.loc[num,0]=1
                        else:
                            break
            else:
                continue
    print(I_to_S_update)

    #z-score
    resized_df_signal_dic_zscore = {}
    for i in df_signal_id:
        temp_noise = resized_df_raw_dic_update[i] - resized_df_signal_dic_update[i]
        # mean = temp_noise.values.mean()
        std = temp_noise.values.std()
        resized_df_signal_dic_zscore[i] = resized_df_signal_dic_update[i]/std

    ## S_to_I_dic -> number s to i -> number cells -> number 40 frames
    S_to_I_dic={}
    for i in range(len(S_to_I_update)):
        S_to_I_signal_dic = {}
        for j in df_signal_id:
            temp_table = resized_df_signal_dic_zscore[j][(S_to_I_update[i]-20):(S_to_I_update[i]+20)]
            S_to_I_signal_dic[j] = temp_table
        S_to_I_dic[i] = S_to_I_signal_dic

    I_to_S_dic={}
    for i in range(len(I_to_S_update)):
        I_to_S_signal_dic = {}
        for j in df_signal_id:
            temp_table = resized_df_signal_dic_update[j][(I_to_S_update[i]-20):(I_to_S_update[i]+20)]
            I_to_S_signal_dic[j] = temp_table
        I_to_S_dic[i] = I_to_S_signal_dic

    ##################################### 
    # calculate event:
    # S—》I20 event*cells*40
    ##################################### 

    S_to_I_df_total_event = {}
    for num in range(len(S_to_I_update)):
        S_to_I_df = pd.DataFrame()
        # S_to_I_df = (S_to_I_dic[num][0] - S_to_I_dic[num][0][:20].values.mean())/S_to_I_dic[num][0][:20].values.std()
        S_to_I_df = S_to_I_dic[num][1]
        S_to_I_df = pd.DataFrame(S_to_I_df).rename(columns={0:0})

        for i in range(1,len(df_signal_id)):
            # S_to_I_zscore_temp = (S_to_I_dic[num][i] - S_to_I_dic[num][i][:20].values.mean())/S_to_I_dic[num][i][:20].values.std()
            S_to_I_zscore_temp = S_to_I_dic[num][i]
            S_to_I_zscore_temp = pd.DataFrame(S_to_I_zscore_temp).rename(columns={0:i})
            S_to_I_df = pd.concat([S_to_I_df,S_to_I_zscore_temp],axis=1)

        S_to_I_df = S_to_I_df.reset_index().drop(columns='index')
        S_to_I_df_total_event[num] = S_to_I_df

    I_to_S_df_total_event = {}
    for num in range(len(I_to_S_update)):
        I_to_S_df = pd.DataFrame()
        # I_to_S_df = (I_to_S_dic[num][0] - I_to_S_dic[num][0][:20].values.mean())/I_to_S_dic[num][0][:20].values.std()
        I_to_S_df = I_to_S_dic[num][1]
        I_to_S_df = pd.DataFrame(I_to_S_df).rename(columns={0:0})

        for i in range(1,len(df_signal_id)):
            # I_to_S_zscore_temp = (I_to_S_dic[num][i] - I_to_S_dic[num][i][:20].values.mean())/I_to_S_dic[num][i][:20].values.std()
            I_to_S_zscore_temp = I_to_S_dic[num][i]
            I_to_S_zscore_temp = pd.DataFrame(I_to_S_zscore_temp).rename(columns={0:i})
            I_to_S_df = pd.concat([I_to_S_df,I_to_S_zscore_temp],axis=1)

        I_to_S_df = I_to_S_df.reset_index().drop(columns='index')
        I_to_S_df_total_event[num] = I_to_S_df

    print('Detected events:')
    print(S_to_I_df_total_event.keys())
    print(I_to_S_df_total_event.keys())

    # 20
    S_to_I_dic_zscore_limit = {}
    for num in range(len(S_to_I_update)):  
        zscore_response_limit = []
        for i in range(len(S_to_I_df_total_event[num].columns)):
            tag_reponse = 0

            prepare_stage = S_to_I_df_total_event[num].loc[:20,i]
            start_stage = S_to_I_df_total_event[num].loc[20:,i]
            t_ks = ks_2samp(prepare_stage,start_stage)
            if t_ks.pvalue < 0.05:
                tag_reponse = 1
            else:
                tag_reponse = 0
            # t_test
            zscore_response_limit.append(tag_reponse)
        
        S_to_I_response = pd.DataFrame(zscore_response_limit).rename(columns={0:'response'})
        S_to_I_dic_zscore_limit[num] = pd.concat([S_to_I_response,S_to_I_df_total_event[num].T],axis=1)
        
    I_to_S_dic_zscore_limit = {}
    for num in range(len(I_to_S_update)):  
        zscore_response_limit = []
        for i in range(len(I_to_S_df_total_event[num].columns)):
            tag_reponse = 0

            prepare_stage = I_to_S_df_total_event[num].loc[:20,i]
            start_stage = I_to_S_df_total_event[num].loc[20:,i]
            t_ks = ks_2samp(prepare_stage,start_stage)
            if t_ks.pvalue < 0.05:
                tag_reponse = 1
            else:
                tag_reponse = 0

            zscore_response_limit.append(tag_reponse)
        
        I_to_S_response = pd.DataFrame(zscore_response_limit).rename(columns={0:'response'})
        I_to_S_dic_zscore_limit[num] = pd.concat([I_to_S_response,I_to_S_df_total_event[num].T],axis=1)
 
    ##################################### 
    # response feature extraction(up/down)
    ####################################
    S_to_I_dic_response_total = {}
    ## a>0, k>20 down
    ## a>0, k<20 up
    ## a<0, k<20 down
    ## a<0, k>20 up
    for num in range(len(S_to_I_update)):
        S_to_I_response = S_to_I_dic_zscore_limit[num][S_to_I_dic_zscore_limit[num]['response']==1].drop(columns=['response'])# only selet significant responses
        S_to_I_upper = pd.DataFrame(np.zeros(len(S_to_I_response),),dtype=int)
        S_to_I_lower = pd.DataFrame(np.zeros(len(S_to_I_response),),dtype=int)
        print('Fitting S_to_I event ' + str(num) + ' ...')
        for i in range(len(S_to_I_response)):
            x = list(range(40))
            y = list(S_to_I_response.iloc[i])
            abc, para = curve_fit(Binary_equation,x,y,maxfev=500000)
            y = list(S_to_I_response.iloc[i])
            liney = Binary_equation(x, *abc)
            a = abc[0]
            b = abc[1]

            # #plot fitting result
            # print(f'a={a}, b={b}')
            # plt.close('all')
            # plt.figure(figsize=(12, 3))
            # plt.plot(x, liney, 'o', label='data')
            # plt.plot(x, y, '-', label='fit')
            # plt.legend()
            # plt.show()
            
            if a > 0:
                if b > 20:
                    # use df.loc[row_indexer, "col"] = values` to avoid panda warning
                    S_to_I_upper.loc[i,0] = 0
                    S_to_I_lower.loc[i,0] = 1
                else:
                    S_to_I_upper.loc[i,0] = 1
                    S_to_I_lower.loc[i,0] = 0
            else:
                if b < 20:
                    S_to_I_upper.loc[i,0] = 0
                    S_to_I_lower.loc[i,0] = 1
                else:
                    S_to_I_upper.loc[i,0] = 1
                    S_to_I_lower.loc[i,0] = 0
        
        S_to_I_upper = S_to_I_upper.rename(columns={0:'upper'})
        S_to_I_lower = S_to_I_lower.rename(columns={0:'lower'})
        S_to_I_response = S_to_I_response.reset_index()
        S_to_I_response = pd.concat([S_to_I_upper,S_to_I_lower,S_to_I_response],axis=1)

        temp = S_to_I_response[['index','upper','lower']]
        temp = temp.set_index('index')

        S_to_I_dic_response_total[num] = pd.concat([S_to_I_dic_zscore_limit[num],temp],axis=1)
        S_to_I_dic_response_total[num] = S_to_I_dic_response_total[num].fillna(0)    

    I_to_S_dic_response_total = {}
    for num in range(len(I_to_S_update)):
        I_to_S_response = I_to_S_dic_zscore_limit[num][I_to_S_dic_zscore_limit[num]['response']==1].drop(columns=['response'])
        I_to_S_upper = pd.DataFrame(np.zeros(len(I_to_S_response),),dtype=int)
        I_to_S_lower = pd.DataFrame(np.zeros(len(I_to_S_response),),dtype=int)
        print('Fitting I_to_S event ' + str(num) + ' ...')
        for i in range(len(I_to_S_response)):
            x = list(range(40))
            y = list(I_to_S_response.iloc[i])
            abc, para = curve_fit(Binary_equation,x,y,maxfev=500000)
            a = abc[0]
            b = abc[1]

            if a > 0:
                if b > 20:
                    I_to_S_upper.loc[i,0] = 0
                    I_to_S_lower.loc[i,0] = 1
                else:
                    I_to_S_upper.loc[i,0] = 1
                    I_to_S_lower.loc[i,0] = 0
            else:
                if b < 20:
                    I_to_S_upper.loc[i,0] = 0
                    I_to_S_lower.loc[i,0] = 1
                else:
                    I_to_S_upper.loc[i,0] = 1
                    I_to_S_lower.loc[i,0] = 0

        I_to_S_upper = I_to_S_upper.rename(columns={0:'upper'})
        I_to_S_lower = I_to_S_lower.rename(columns={0:'lower'})
        I_to_S_response = I_to_S_response.reset_index()
        I_to_S_response = pd.concat([I_to_S_upper,I_to_S_lower,I_to_S_response],axis=1)

        temp = I_to_S_response[['index','upper','lower']]
        temp = temp.set_index('index')

        I_to_S_dic_response_total[num] = pd.concat([I_to_S_dic_zscore_limit[num],temp],axis=1)
        I_to_S_dic_response_total[num] = I_to_S_dic_response_total[num].fillna(0)    

    #result combine
    S_to_I_upper_total = pd.DataFrame(np.zeros(len(df_signal_id),),dtype=int)
    S_to_I_lower_total = pd.DataFrame(np.zeros(len(df_signal_id),),dtype=int)
    for i in range(len(S_to_I_update)):
        S_to_I_upper_total[0] = S_to_I_upper_total[0] + S_to_I_dic_response_total[i].upper
        S_to_I_lower_total[0] = S_to_I_lower_total[0] + S_to_I_dic_response_total[i].lower

    I_to_S_upper_total = pd.DataFrame(np.zeros(len(df_signal_id),),dtype=int)
    I_to_S_lower_total = pd.DataFrame(np.zeros(len(df_signal_id),),dtype=int)
    for i in range(len(I_to_S_update)):
        I_to_S_upper_total[0] = I_to_S_upper_total[0] + I_to_S_dic_response_total[i].upper
        I_to_S_lower_total[0] = I_to_S_lower_total[0] + I_to_S_dic_response_total[i].lower

    S_to_I_upper_total = S_to_I_upper_total.rename(columns={0:'S_to_I_up'})
    S_to_I_lower_total = S_to_I_lower_total.rename(columns={0:'S_to_I_down'})
    I_to_S_upper_total = I_to_S_upper_total.rename(columns={0:'I_to_S_up'})
    I_to_S_lower_total = I_to_S_lower_total.rename(columns={0:'I_to_S_down'})
    analysis_total = pd.concat([S_to_I_upper_total,S_to_I_lower_total,I_to_S_upper_total,I_to_S_lower_total],axis=1)
    print('Final preview:')
    print(analysis_total)

    ###########
    #FileSave
    ###########
    # save_path = r'D:/RQ/M2 Project/summary/20240829 m2_v9_lollipop_chart'
    for i in range(len(S_to_I_update)):
        S_to_I_dic_response_total[i].to_csv(save_path+'/'+'S_to_I_zscore_event_' + str(i) + '.csv')

    for i in range(len(I_to_S_update)):
        I_to_S_dic_response_total[i].to_csv(save_path+'/'+'I_to_S_zscore_event_' + str(i) + '.csv')

    analysis_total.to_csv(save_path+'/'+'analysis_total.csv')
    print('Data saved to :'+ save_path+'/'+'analysis_total.csv')